In [1]:
!pip install uv && uv pip install edsl && uv pip install beep


import random
import os

  Obtaining dependency information for uv from https://files.pythonhosted.org/packages/03/00/9a070fbeb77516ed8ba1a4ddf9a83836c8b579355acde185480c0958d6f2/uv-0.6.4-py3-none-macosx_11_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 16.1 MB/s eta 0:00:0000:0100:01
Using Python 3.11.5 environment at: /Users/avshah/anaconda3
Resolved 149 packages in 954ms                                       
⠙ Preparing packages... (0/12)                                                  
⠙ Preparing packages... (0/12)---     0 B/76.71 KiB                     
⠙ Preparing packages... (0/12)--- 14.88 KiB/76.71 KiB                   
httpcore   ------------------------------ 14.88 KiB/76.71 KiB
⠙ Preparing packages... (0/12)---     0 B/236.74 KiB                    
httpcore   ------------------------------ 14.88 KiB/76.71 KiB
rich       ------------------------------     0 B/236.74 KiB
⠙ Preparing packages... (0/12)---     0 B/1.17 MiB                      
httpcore   ---------

In [3]:
import os

os.environ['EXPECTED_PARROT_API_KEY'] = 'jDJypShYzhFCynBSEScswpaHgoFiuJHq4XGwUlKNQH8'

In [ ]:
from edsl import Scenario, Survey, QuestionFreeText, QuestionNumerical
import random
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------
# Configuration: number of bidders and rounds
num_bidders = 5
num_rounds = 5

# Global history list: each round’s data is stored here.
# Each entry is a dictionary with keys:
#   "round": round number,
#   "plans": dict mapping bidder -> plan,
#   "values": dict mapping bidder -> value,
#   "bids": dict mapping bidder -> bid,
#   "winner": winning bidder,
#   "payment": winner’s payment,
#   "outcomes": dict mapping bidder -> reflection
history = []

# For plotting
all_rounds_data = []

# ------------------------------
# Helper: Build personalized history string for a bidder
def build_history_string(bidder, history):
    """For a given bidder, build a string summarizing all past rounds."""
    lines = []
    for r in history:
        line = (f"Round {r['round']}: "
                f"Your plan: {r['plans'][bidder]}. "
                f"Your value: ${r['values'][bidder]}. "
                f"All bids: {r['bids']}. "
                f"Winner: {r['winner']} pays ${r['payment']}. "
                f"Your reflection: {r['outcomes'][bidder]}")
        lines.append(line)
    return "\n".join(lines) if lines else "No history! This is the first round."

# ------------------------------
# 1) Bidder Generation
def generate_bidder_names(num_bidders):
    """
    Generate a list of famous economist bidder names using EDSL.
    """
    q_names = QuestionFreeText(
        question_name="bidder_names",
        question_text="""Generate a list of {{ num_bidders }} random bidder names,
        where each name is a single word and starts with a different letter from A, B, C, etc.
        Return **only** the names in a comma-separated string with no extra text, e.g., "Anand, Betty, Chen"."""
    )
    scenario = Scenario({"num_bidders": num_bidders})
    survey = Survey([q_names])
    results = survey.by(scenario).run()
    bidder_names_str = results.select("bidder_names").first().strip("[]").replace("'", "").replace('"', '')
    bidder_names = [name.strip() for name in bidder_names_str.split(",")]
    if len(bidder_names) != num_bidders:
        raise ValueError(f"Expected {num_bidders} names, got {len(bidder_names)}: {bidder_names}")
    return bidder_names

bidder_names = generate_bidder_names(num_bidders)

# ------------------------------
# 2) Iterate over rounds
for round_num in range(1, num_rounds + 1):
    print(f"Running round {round_num}...")

    # Generate scenario data for current round:
    # assign a true value for each bidder
    scenario_data = {
        "round": round_num,
        **{f"value_{name}": random.randint(0, 99) for name in bidder_names}
    }
    
    # For each bidder, build their personalized history string (from previous rounds)
    for bidder in bidder_names:
        scenario_data[f"{bidder}_history"] = build_history_string(bidder, history)
    
    # Build the base scenario
    scenario = Scenario(scenario_data)
    
    # --- First Survey: Get bidding plans for the current round ---
    plan_questions = []
    for bidder in bidder_names:
        q_plan = QuestionFreeText(
            question_name=f"plan_{bidder}",
            question_text=f"""
Your name is {bidder}.
In this auction, you will compete against {num_bidders - 1} other bidders.
At the start of this round, you receive your private value (a random number between $0 and $99).
You will later submit a bid.
Before that, here is your history from previous rounds:
{scenario_data[f"{bidder}_history"]}
In one sentence, please come up with a plan for this round.
            """
        )
        plan_questions.append(q_plan)
    survey_plans = Survey(plan_questions)
    results_plans = survey_plans.by(scenario).run()
    
    # Update scenario_data with the plan answers for the current round
    for bidder in bidder_names:
        plan_answer = results_plans.select(f"plan_{bidder}").first()
        scenario_data[f"plan_{bidder}"] = plan_answer
    
    # --- Second Survey: Ask for bids using the resolved current plan and value ---
    bid_questions = []
    for bidder in bidder_names:
        # Make sure to convert the true value (an int) to a string
        q_bid_text = (
            "Your name is " + bidder + ".\n" +
            "Your plan for this round was: " + scenario_data[f"plan_{bidder}"] + ".\n" +
            "Your private value for the prize is: " + str(scenario_data[f"value_{bidder}"]) + ".\n" +
            "What is your bid?"
        )
        q_bid = QuestionNumerical(
            question_name=f"bid_{bidder}",
            question_text=q_bid_text,
            min_value=0,
            max_value=99
        )
        bid_questions.append(q_bid)
    survey_bids = Survey(bid_questions)
    results_bids = survey_bids.by(scenario).run()
    
    # Extract bids from the survey results
    bids_df = results_bids.select(*[f"bid_{bidder}" for bidder in bidder_names]).to_pandas()
    bids = {bidder: bids_df[f"answer.bid_{bidder}"].iloc[0] for bidder in bidder_names}
    
    # Determine winner and payment (using second-highest bid)
    sorted_bids = sorted(bids.values(), reverse=True)
    winning_bid = sorted_bids[0]
    second_highest_bid = sorted_bids[1] if len(sorted_bids) > 1 else 0
    winner = [name for name, bid in bids.items() if bid == winning_bid][0] #rn not implementing random tie-breaker, just choosing first. \avs{come back and fix this}
    
    # --- Third Survey: Outcome reflections ---
    outcome_questions = []
    for bidder in bidder_names:
        q_outcome = QuestionFreeText(
            question_name=f"outcome_{bidder}",
            question_text=f"""
Your name is {bidder}.
Your private value for the prize was: {scenario_data[f"value_{bidder}"]}.
You submitted a bid of: {bids[bidder]}.
All bids in this round were: {bids}.
What happens in this round? How do you feel about your bid and the outcome?
Could you have done better?
            """
        )
        outcome_questions.append(q_outcome)
    survey_outcomes = Survey(outcome_questions)
    results_outcomes = survey_outcomes.by(scenario).run()
    
    # Extract reflections (outcomes)
    outcomes_df = results_outcomes.select(*[f"outcome_{bidder}" for bidder in bidder_names]).to_pandas()
    outcomes = {bidder: outcomes_df[f"answer.outcome_{bidder}"].iloc[0] for bidder in bidder_names}
    
    # --- Update history with all data from the current round ---
    round_record = {
        "round": round_num,
        "plans": {bidder: scenario_data[f"plan_{bidder}"] for bidder in bidder_names},
        "values": {bidder: scenario_data[f"value_{bidder}"] for bidder in bidder_names},
        "bids": bids,
        "winner": winner,
        "payment": second_highest_bid,
        "outcomes": outcomes
    }
    history.append(round_record)
    
    # For plotting, store summary data for each bidder for this round
    for bidder in bidder_names:
        all_rounds_data.append({
            "round": round_num,
            "bidder": bidder,
            "value": scenario_data[f"value_{bidder}"],
            "bid": bids[bidder]
        })

# ------------------------------
# After all rounds, build a DataFrame and plot summaries

df = pd.DataFrame(all_rounds_data)

# Plot bids and values over time
plt.figure(figsize=(12, 6))
for bidder in bidder_names:
    bidder_data = df[df["bidder"] == bidder]
    plt.plot(bidder_data["round"], bidder_data["bid"], label=f"{bidder} Bid", marker="o")
    plt.plot(bidder_data["round"], bidder_data["value"], label=f"{bidder} Value", marker="x", linestyle="--")
plt.xlabel("Round")
plt.ylabel("Amount ($)")
plt.title("Bids and Values Over Time")
plt.legend()
plt.grid()
plt.show()

# Plot bid-to-value ratio
plt.figure(figsize=(12, 6))
for bidder in bidder_names:
    bidder_data = df[df["bidder"] == bidder].copy()
    bidder_data["bid_ratio"] = [bid / max(val, 1) * 100 for bid, val in zip(bidder_data["bid"], bidder_data["value"])]
    plt.plot(bidder_data["round"], bidder_data["bid_ratio"], label=f"{bidder} Bid/Value %", marker="o")
plt.xlabel("Round")
plt.ylabel("Bid as % of Value")
plt.title("Bid-to-Value Ratio Over Time")
plt.legend()
plt.grid()
plt.show()


TypeError: Jobs.run() got an unexpected keyword argument 'verbose'